# GSM8K LoRA Study

### Fine-tuning and prompting for mathematical reasoning

**Question:** Can a small instruction-tuned language model solve more math word problems after lightweight fine-tuning?

I evaluated Qwen2.5-1.5B-Instruct on GSM8K, trained LoRA adapters, and compared zero-shot and three-shot prompting.
The best recorded configuration answered **47 of 100 questions correctly**, versus **36 of 100** for the base model.
This is an exploratory result on a small sample, not a claim about the full benchmark.

This notebook presents the experiment, results, and lessons first. An optional rerun section follows.
It is a newly written portfolio walkthrough; original assignment documents, helper templates, and private metadata are excluded.

## 1. Method

- **Model:** Qwen2.5-1.5B-Instruct, an instruction-tuned causal language model.
- **Dataset:** GSM8K math word problems; the historical runs used the first 100 test examples.
- **Adaptation:** LoRA learns low-rank updates to attention projections while keeping the base weights frozen.
- **Comparison:** Base model, adapters trained on 1,000 and 2,000 examples, and three-shot prompting.
- **Metric:** Exact numeric answer accuracy, with a final-number fallback when no boxed answer was present.

Historical training settings differed: the 1k run used rank 8, alpha 16, one epoch, and learning rate 0.0002;
the 2k run used rank 16, alpha 32, two epochs, and learning rate 0.0001. Therefore, their difference cannot
be attributed to training-set size alone. Few-shot examples were taken from training data.

In [ ]:
from pathlib import Path
import csv
import json
import gc
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "results" / "historical_summary.csv").is_file():
    raise RuntimeError("Start Jupyter from the repository folder")
with (ROOT / "results" / "historical_summary.csv").open(encoding="utf-8", newline="") as handle:
    historical = list(csv.DictReader(handle))
for row in historical:
    print(f"{row['experiment']}: {row['correct']}/{row['total']}")

## 2. Recorded results

These aggregates were checked against five saved evaluation files. They are **historical results**, not
measurements from the fresh rerun code below. Raw coursework exports and model responses are not distributed.

In [ ]:
labels = [row["experiment"] for row in historical]
accuracy = [100 * int(row["correct"]) / int(row["total"]) for row in historical]
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
colors = ["#64748b", "#5b8db8", "#39739d", "#94a3b8", "#0f766e"]
bars = ax.barh(labels, accuracy, color=colors)
ax.bar_label(bars, labels=[f"{value:.0f}%" for value in accuracy], padding=6)
ax.set(xlim=(0, 100), xlabel="Accuracy on 100 test questions (%)",
       title="LoRA and prompting: recorded experiment results")
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)
ax.set_axisbelow(True)
ax.grid(axis="x", alpha=0.15)
plt.show()

## 3. What changed, and what did not?

The combined 2k adapter plus three-shot prompt improved accuracy by **11 percentage points** over the base zero-shot model.
Three-shot prompting alone did not help the base model in this sample (34% versus 36%).
The saved records were paired by question and index before counting the following outcomes:

| Base → adapted three-shot outcome | Questions |
| --- | ---: |
| Correct → correct | 21 |
| Wrong → correct | 26 |
| Correct → wrong | 15 |
| Wrong → wrong | 38 |

**Interpretation:** The net gain hides regressions. The adapted configuration corrected 26 mistakes but introduced 15 others.
This does not establish statistical significance or isolate the effect of LoRA from prompting and changed training settings.

## 4. Two concrete error examples

These are concise paraphrases of public GSM8K test problems and the saved model behavior, not raw notebook outputs.

**Arithmetic corrected — test row 9.** A pay calculation combines 40 regular hours at $10 with 5 overtime hours at 1.2 times the rate.
The base model set up the calculation correctly but evaluated $5 \times 12$ as 10, yielding 410.
The adapted three-shot model computed 60 in overtime and reached the correct total of **460**.

**Problem interpretation regressed — test row 0.** A seller has 16 eggs, uses 3 and 4, and sells the remainder for $2 each.
The base model correctly computed $(16-3-4)\times2=18$.
The adapted model incorrectly began with $16\times3$ and ended at **82** rather than **18**.

**Lesson:** A consistent answer format and fluent reasoning do not guarantee correct arithmetic or faithful use of the problem's facts.
These two cases illustrate behavior; they are not a systematic classification of all errors.

## 5. Limitations and next experiment

- Only 100 test questions were evaluated. These were exploratory test-set comparisons, not a fresh final holdout selected after tuning.
- Training size, rank, epochs, and learning rate changed together in the historical 1k/2k comparison.
- The original later prompt-sweep section had a Python backslash escape bug; its scores are excluded here.
- Historical metrics used the original scorer. The rerun scorer below is stricter and also reports boxed-answer compliance.
- No full-model retraining has been performed to reproduce these numbers in this cleaned project.

The next experiment should hold training settings fixed, select prompts on development data, inspect regressions,
and evaluate the chosen configuration on held-out test questions. Repeat across seeds before making broader claims.

## 6. Optional: run a fresh experiment

The following code is self-contained. It is a **revised experiment**, not a bit-for-bit replay of the historical runs:
it uses disjoint development/demonstration/training pools, fresh prompts, fixed adapter settings, and stricter scoring.
Historical and fresh results are kept separate.

Default execution only displays the historical analysis and checks helper functions. To download the public model/dataset
and run inference/training, set `RUN_EXPERIMENTS = True`. Full training needs suitable hardware; 4 GB VRAM has not been validated.
CPU runs are supported but slow. There is no quantization or offloading. Each run gets a new output directory.

In [ ]:
RUN_EXPERIMENTS = False
RUN_NAME = "experiment-01"  # Use a new name for each run.
TRAIN_SIZE = 1000            # Try 2000 in a separate run with other settings fixed.
EVAL_SIZE = 100
SEED = 42
DEVICE = "auto"
RUN_FINAL_TEST = False       # Enable only after selecting settings on development data.

### Answer scoring

Compare supported numbers exactly; never accept an empty answer. Report numeric correctness and correct boxed answers separately.
The final-number fallback can still select an incidental number, so inspect predictions as well as the score.

In [ ]:
"""Conservative exact-number scoring. No expression evaluation."""

import re
from fractions import Fraction


def boxed_content(text: str) -> str | None:
    """Read the final box, including nested braces; reject an unclosed box."""
    start = text.rfind(r"\boxed{")
    if start < 0:
        return None
    start += len(r"\boxed{")
    depth = 1
    for end in range(start, len(text)):
        if text[end] == "{":
            depth += 1
        elif text[end] == "}":
            depth -= 1
        if depth == 0:
            return text[start:end]
    return None


def number(text: str | None) -> Fraction | None:
    if text is None:
        return None
    value = text.strip().replace("−", "-")
    latex = re.fullmatch(r"([+-]?)\\(?:d?frac)\{([+-]?\d+)\}\{([+-]?\d+)\}", value)
    if latex:
        sign, numerator, denominator = latex.groups()
        value = f"{sign}{numerator}/{denominator}"
    # Commas must form proper groups; do not silently turn 1,5 into 15.
    if "," in value:
        if not re.fullmatch(r"[+-]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", value):
            return None
        value = value.replace(",", "")
    if not re.fullmatch(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d{1,3})?|[+-]?\d+/[+-]?\d+", value):
        return None
    if len(value) > 128:
        return None
    try:
        return Fraction(value)
    except (ValueError, ZeroDivisionError):
        return None


def score(response: str, expected: str) -> dict:
    """Numeric accuracy permits a final-number fallback; strict accuracy needs a box."""
    target = number(expected)
    if target is None:
        raise ValueError("Reference answer is not a supported number")
    content = boxed_content(response)
    has_box = r"\boxed{" in response
    candidate = content
    if not has_box:
        matches = re.findall(r"[+-]?(?:\d[\d,]*(?:\.\d+)?|\.\d+)(?:/[+-]?\d+)?(?:%|[eE][+-]?\d+)?", response)
        candidate = matches[-1] if matches else None
    parsed = number(candidate)
    correct = parsed is not None and parsed == target
    return {
        "answer": str(parsed) if parsed is not None else None,
        "correct": correct,
        "boxed": content is not None and parsed is not None,
        "strict_correct": correct and content is not None,
        "invalid": parsed is None,
    }


def summarize(records: list[dict]) -> dict:
    if not records:
        raise ValueError("Cannot summarize an empty evaluation")
    total = len(records)
    return {"total": total, **{
        key: sum(bool(row[key]) for row in records) / total
        for key in ("correct", "strict_correct", "boxed", "invalid")
    }}

In [ ]:
assert score(r"The result is \boxed{7}", "7")["strict_correct"]
assert not score(r"\boxed{}", "0")["correct"]
assert not score(r"\boxed{7", "7")["correct"]
assert score(r"\boxed{\frac{1}{2}}", "0.5")["correct"]
assert not score("50%", "50")["correct"]
print("Scoring checks passed")

### Prompts and disjoint data pools

Reserve the first 200 training rows for development and the next three for demonstrations. Fine-tuning uses the remaining
training rows. Shuffle each pool with the chosen seed; record selected row IDs. The original dataset test split is reserved for final evaluation.

In [ ]:
"""Fresh prompt templates, shared by training and inference."""

PROMPTS = {
    "standard": r"Solve the math problem. Explain your calculation briefly and finish with \boxed{answer}.",
    "check": r"Solve the math problem and check your arithmetic. Finish with \boxed{answer}.",
    "concise": r"Solve the math problem using only the necessary calculations. Finish with \boxed{answer}.",
}


def completion(answer: str) -> str:
    reasoning, separator, final = answer.rpartition("####")
    if not separator or not final.strip():
        raise ValueError("Expected a GSM8K answer with a #### delimiter")
    return reasoning.strip() + "\n" + r"\boxed{" + final.strip() + "}"


def messages(question: str, prompt: str, examples: list[dict] | None = None) -> list[dict]:
    result = [{"role": "system", "content": PROMPTS[prompt]}]
    for example in examples or []:
        result.extend([
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": completion(example["answer"])},
        ])
    return result + [{"role": "user", "content": question}]

"""Stable, disjoint splits: first 200 train rows are development-only."""

import random

DEV_SIZE = 200
SHOT_SIZE = 3
DATASET = "openai/gsm8k"


def select_indices(length: int, split: str, limit: int, seed: int) -> list[int]:
    pools = {
        "train": range(DEV_SIZE + SHOT_SIZE, length),
        "dev": range(min(DEV_SIZE, length)),
        "shots": range(DEV_SIZE, min(DEV_SIZE + SHOT_SIZE, length)),
        "test": range(length),
    }
    indices = list(pools[split])
    if limit < 1 or limit > len(indices):
        raise ValueError(f"Requested {limit} rows from {split}; available: {len(indices)}")
    if split != "shots":
        random.Random(seed).shuffle(indices)
    return indices[:limit]


def load_rows(split: str, limit: int, seed: int, revision: str) -> tuple[list[dict], dict]:
    from datasets import load_dataset
    dataset = load_dataset(DATASET, "main", split="test" if split == "test" else "train", revision=revision)
    indices = select_indices(len(dataset), split, limit, seed)
    rows = [{"id": i, **dataset[i]} for i in indices]
    return rows, {"dataset": DATASET, "revision": revision, "fingerprint": dataset._fingerprint,
                  "split": split, "row_ids": indices, "seed": seed}

In [ ]:
training_ids = set(select_indices(7473, "train", 2000, SEED))
development_ids = set(select_indices(7473, "dev", 200, SEED))
shot_ids = set(select_indices(7473, "shots", 3, SEED))
assert not (training_ids & development_ids or training_ids & shot_ids or development_ids & shot_ids)
assert set(select_indices(7473, "train", 1000, SEED)) <= training_ids
print("Split checks passed")

### Model loading and evaluation

Load one model at a time, use greedy decoding, and score only generated continuation tokens. Resolve upstream versions
to commits before loading. Adapter metadata must match the selected base model version.

In [ ]:
"""Lazy model loading and metadata without usernames or machine paths."""

import importlib.metadata
import json
from pathlib import Path

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


def dependencies() -> dict:
    result = {}
    for name in ("torch", "transformers", "peft", "datasets", "accelerate"):
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            pass
    return result


def precision(device: str, dtype: str):
    import torch
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA is unavailable; install a CUDA PyTorch build or use --device cpu")
    if dtype == "auto":
        dtype = "bfloat16" if device == "cuda" and torch.cuda.is_bf16_supported() else "float32"
    if dtype == "bfloat16" and (device != "cuda" or not torch.cuda.is_bf16_supported()):
        raise ValueError("bfloat16 requires a supported CUDA device; use float32")
    return device, getattr(torch, dtype)


def load_model(model_id: str, revision: str, device: str, dtype: str, adapter: str | None = None):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    device, torch_dtype = precision(device, dtype)
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
    if not tokenizer.chat_template:
        raise ValueError("The model tokenizer must provide a chat template")
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, revision=revision, torch_dtype=torch_dtype, trust_remote_code=False,
        attn_implementation="eager",
    ).to(device)
    if adapter:
        metadata_path = Path(adapter) / "experiment.json"
        if not metadata_path.is_file():
            raise ValueError("Adapter must be produced by gsm8k-lab train (missing experiment.json)")
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        resolved = getattr(model.config, "_commit_hash", None)
        if metadata["model"] != model_id or metadata["model_revision"] != resolved:
            raise ValueError("Adapter base model/revision does not match the selected model")
        model = PeftModel.from_pretrained(model, adapter)
    return model, tokenizer


def new_output(path: str) -> Path:
    destination = Path(path)
    destination.mkdir(parents=True, exist_ok=False)
    return destination


def write_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

"""One model per process; deterministic greedy decoding."""



def evaluate(model, tokenizer, rows, prompt="standard", examples=None, max_new_tokens=512, batch_size=1):
    import torch
    tokenizer.padding_side = "left"
    model.eval()
    records = []
    for offset in range(0, len(rows), batch_size):
        batch = rows[offset:offset + batch_size]
        rendered = [tokenizer.apply_chat_template(
            messages(row["question"], prompt, examples), tokenize=False, add_generation_prompt=True,
        ) for row in batch]
        tokens = tokenizer(rendered, padding=True, add_special_tokens=False,
                           return_token_type_ids=False, return_tensors="pt").to(model.device)
        context = getattr(model.config, "max_position_embeddings", None)
        if context and tokens.input_ids.shape[1] + max_new_tokens > context:
            raise ValueError("Prompt plus response exceeds model context; reduce shots or response length")
        with torch.inference_mode():
            generated = model.generate(**tokens, max_new_tokens=max_new_tokens, do_sample=False,
                                       pad_token_id=tokenizer.pad_token_id)
        responses = tokenizer.batch_decode(generated[:, tokens.input_ids.shape[1]:], skip_special_tokens=True)
        for row, response in zip(batch, responses):
            expected = row["answer"].rsplit("####", 1)[-1].strip()
            records.append({"id": row["id"], "response": response, "expected": expected,
                            **score(response, expected)})
    return records, summarize(records)

### Completion-only LoRA training

Only assistant tokens contribute to training loss; prompt and padding tokens are masked. Overlength examples are skipped and counted.
The defaults use rank 8, alpha 16, one epoch, learning rate 0.0002, and gradient accumulation of 8 microbatches.

In [ ]:
"""Completion-only supervised LoRA training with Hugging Face Trainer."""

import math



def encode_example(tokenizer, row, max_length, prompt="standard"):
    conversation = messages(row["question"], prompt)
    prefix = tokenizer.apply_chat_template(conversation, tokenize=True, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(
        conversation + [{"role": "assistant", "content": completion(row["answer"])}],
        tokenize=True, add_generation_prompt=False,
    )
    if full[:len(prefix)] != prefix:
        raise ValueError("Chat template does not preserve the assistant prefix")
    # Reject rather than train on a clipped rationale or missing final answer.
    if len(full) > max_length:
        return None
    if len(full) <= len(prefix):
        raise ValueError("Example contains no assistant tokens")
    return {"input_ids": full, "attention_mask": [1] * len(full),
            "labels": [-100] * len(prefix) + full[len(prefix):]}


class CompletionCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        import torch
        width = max(len(feature["input_ids"]) for feature in features)
        return {key: torch.tensor([
            feature[key] + [pad] * (width - len(feature[key])) for feature in features
        ]) for key, pad in (("input_ids", self.pad_token_id), ("attention_mask", 0), ("labels", -100))}


def train(model, tokenizer, rows, output, config):
    import torch
    from peft import LoraConfig, get_peft_model
    from transformers import Trainer, TrainingArguments
    tokenizer.padding_side = "right"
    encoded = [encode_example(tokenizer, row, config["max_length"], config["prompt"]) for row in rows]
    features = [row for row in encoded if row is not None]
    if not features:
        raise ValueError("All training rows exceed max_length; increase the limit")
    model = get_peft_model(model, LoraConfig(
        task_type="CAUSAL_LM", r=config["rank"], lora_alpha=2 * config["rank"], lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none",
    ))
    model.config.use_cache = False
    model.enable_input_require_grads()
    step_budget = config["max_steps"] if config["max_steps"] > 0 else (
        math.ceil(len(features) / (config["batch_size"] * config["accumulation"])) * config["epochs"]
    )
    args = TrainingArguments(
        output_dir=str(output / "checkpoints"), per_device_train_batch_size=config["batch_size"],
        gradient_accumulation_steps=config["accumulation"], num_train_epochs=config["epochs"],
        max_steps=config["max_steps"], learning_rate=config["learning_rate"],
        lr_scheduler_type="cosine", warmup_ratio=0.0 if step_budget == 1 else 0.05,
        seed=config["seed"], data_seed=config["seed"],
        bf16=model.dtype == torch.bfloat16, fp16=False, use_cpu=model.device.type == "cpu",
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        save_strategy="no", logging_strategy="steps", logging_steps=1, report_to=[],
        remove_unused_columns=False, dataloader_pin_memory=model.device.type == "cuda",
    )
    trainer = Trainer(model=model, args=args, train_dataset=features,
                      data_collator=CompletionCollator(tokenizer.pad_token_id))
    result = trainer.train()
    adapter = output / "adapter"
    model.save_pretrained(adapter, safe_serialization=True)
    tokenizer.save_pretrained(adapter)
    return {"used_rows": len(features), "skipped_long_rows": len(rows) - len(features),
            "used_row_ids": [row["id"] for row, encoding in zip(rows, encoded) if encoding is not None],
            "optimizer_steps": result.global_step, "training_loss": result.training_loss}

### Run baseline → adapter → prompting comparisons

This cell deliberately runs each model sequentially to release memory between stages. It saves settings and evaluation outputs
under the ignored `runs/` folder. It does not send run artifacts to GitHub or an experiment-tracking service.

In [ ]:
if RUN_EXPERIMENTS:
    import torch
    from huggingface_hub import HfApi
    from transformers import set_seed

    set_seed(SEED)
    output = new_output(str(ROOT / "runs" / RUN_NAME))
    model_revision = HfApi().model_info(MODEL).sha
    dataset_revision = HfApi().dataset_info(DATASET).sha
    train_rows, train_info = load_rows("train", TRAIN_SIZE, SEED, dataset_revision)
    dev_rows, dev_info = load_rows("dev", EVAL_SIZE, SEED, dataset_revision)
    shots, _ = load_rows("shots", 3, SEED, dataset_revision)
    metadata = {"model": MODEL, "model_revision": model_revision,
                "dataset_revision": dataset_revision, "seed": SEED, "versions": dependencies()}
    settings = dict(max_length=1024, prompt="standard", rank=8, batch_size=1,
                    accumulation=8, epochs=1, max_steps=-1, learning_rate=0.0002, seed=SEED)
    write_json(output / "config.json", {**metadata, "training": settings, "data": train_info})

    def release_memory():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def measure(name, rows, data_info, adapter=None, examples=None):
        model, tokenizer = load_model(MODEL, model_revision, DEVICE, "auto", adapter)
        try:
            records, metrics = evaluate(model, tokenizer, rows, examples=examples)
            write_json(output / f"{name}.json", {**metadata, "data": data_info, "metrics": metrics,
                       "prompt": "standard", "shots": len(examples or []), "max_new_tokens": 512,
                       "dtype": str(model.dtype), "device": model.device.type, "records": records})
            print(name, metrics)
            return metrics
        finally:
            del model, tokenizer
            release_memory()

    fresh_results = {"base-dev": measure("base-dev", dev_rows, dev_info)}
    model, tokenizer = load_model(MODEL, model_revision, DEVICE, "auto")
    try:
        training_metrics = train(model, tokenizer, train_rows, output, settings)
        write_json(output / "adapter" / "experiment.json", {**metadata, "training": settings,
                   "data": train_info, "metrics": training_metrics, "dtype": str(model.dtype), "device": model.device.type})
    finally:
        del model, tokenizer
        release_memory()
    adapter = str(output / "adapter")
    fresh_results["adapter-dev"] = measure("adapter-dev", dev_rows, dev_info, adapter)
    fresh_results["adapter-3shot-dev"] = measure("adapter-3shot-dev", dev_rows, dev_info, adapter, shots)
    if RUN_FINAL_TEST:
        test_rows, test_info = load_rows("test", EVAL_SIZE, SEED, dataset_revision)
        fresh_results["base-test"] = measure("base-test", test_rows, test_info)
        fresh_results["adapter-test"] = measure("adapter-test", test_rows, test_info, adapter)
else:
    print("Review mode: model downloads, training, and inference are disabled.")

## References and sharing

- [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
- [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k) and [paper](https://arxiv.org/abs/2110.14168)
- [LoRA paper](https://arxiv.org/abs/2106.09685) and [PEFT](https://huggingface.co/docs/peft/en/index)
- [Transformers Trainer](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer)

The original experiments used supplied educational helpers and these upstream libraries. This cleaned notebook uses newly written
helpers rather than redistributing those templates. Models and data retain their upstream licenses and are downloaded at runtime.
Clear outputs and inspect metadata before sharing a modified notebook. Generated runs are ignored by Git.